# 05 · ALS và vector phiên (ticket #14)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/als_{val,test}_run0.parquet`
- `item_embeddings/als.npy`, `item_embeddings/als_tracks.parquet`
- `als_params.json`: tham số đã chọn và NDCG@10 trên val của cả lưới

Lưới chọn trên val: factors {64, 128} × regularization {0,01; 0,1} × decay {0,5; 0,7; 0,9; 1,0} (16 cấu hình, 4 lần học ALS). Chọn theo NDCG@10.

Bật **Internet** (clone repo, `pip install implicit`). Chỉ cần CPU. Ước tính 20–30 phút.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #14 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
# implicit không có sẵn trên Kaggle (#2)
!pip install -q implicit==0.7.3

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.als import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # chọn tham số trên val, xuất file, có validate_dataset bên trong
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lưới trên val, xếp theo NDCG@10, so với item-kNN liền kề (#9, notebook 04)
import pandas as pd

ITEMKNN_NDCG = 0.1297
grid = pd.DataFrame(report["grid"]).sort_values("ndcg", ascending=False)
print(grid.round(4).to_string(index=False))
best = report["val"]
print(f"\nChọn: {report['params']} | NDCG@10 val {best['ndcg']:.4f} so với item-kNN {ITEMKNN_NDCG} ({best['ndcg'] / ITEMKNN_NDCG - 1:+.1%})")

## Đăng output

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Ở tab Output của version vừa chạy → **New Dataset** tên `mixnext-als`: `candidates/` dùng cùng các file trong `mixnext-candidates`, `item_embeddings/` là embedding cho app (#20) và cho sinh chuỗi (#18).

## Kết quả trên val (chưa chạy)

Dán bảng lưới và dòng "Chọn: …" vào đây sau khi chạy trên Kaggle.